# Using [Cortex-JS Compute Engine](https://mathlive.io/compute-engine/) in Raku

Anton Antonov   
March 2026 

---

## Setup

In [1]:
use CortexJS;
use LaTeX::Grammar;

---

## Conversion of LaTeX to MathJSON

The LaTeX expressions can be converted to MathJSON using the Cortex-JS functions or by the Raku package ["LaTeX::Grammar"](https://raku.land/zef:antononcube/LaTeX::Grammar). Here is an example with the latter:

In [2]:
latex-interpret('\sqrt{4 * x^2 + 12 * x + 9}')

["Root",["Add",["Add",["Multiply",4,["Power","x",2]],["Multiply",12,"x"]],9],2]

There are some notable difference between the two converters:

In [36]:
#% html
[
    '4 x^2 + 12 x + 9',
    '4 \times x^2 + 12 \times x + 9',
    '4 * x^2 + 12 * x + 9',
    '\sqrt{4x^2 + 12x + 9}',
    '\sqrt{4 * x^2 + 12 * x + 9}',
]
andthen $_.map({ %(expr => "latex«$_»", Cortex-JS => parse-latex($_).raku, 'LaTeX::Grammar' => latex-interpret($_)) })
andthen .&to-html(field-names => <expr Cortex-JS LaTeX::Grammar>, align => 'left')
andthen .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g)

expr,Cortex-JS,LaTeX::Grammar
4 x2+12 x+9,"$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","[""Add"",[""Add"",[""Power"",""4 x"",2],""12 x""],9]"
4×x2+12×x+9,"$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9]"
4×x2+12×x+9,"$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9]"
4x2+12x+9,"$[""Sqrt"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]]","[""Root"",[""Add"",[""Add"",[""Power"",""4x"",2],""12x""],9],2]"
4×x2+12×x+9,"$[""Sqrt"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]]","[""Root"",[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9],2]"


----

## Algebraic operations

In [4]:
'e^{i\\pi}'
==> parse-latex()
==> evaluate()

-1

In [5]:
'(a + b)^2'==> 
parse-latex()==> 
expand()==>
to-latex(:!bracketed)

a^2+b^2+2ab

Simplify an expression:

In [6]:
#%markdown
my $expr = parse-latex('3x^2 + 2x^2 + x + 5');
"{to-latex($expr)} = {to-latex(simplify($expr))}";

$2x^2+3x^2+x+5$ = $5x^2+x+5$

Convert to MathJSON first then simplify:

In [7]:
#%markdown
'\sqrt{3x^2 + 2x^2 + x + 5}'
==> parse-latex()
==> simplify()
==> to-latex()

$\sqrt{5x^2+x+5}$

----

## Solve equations

In [8]:
'x^2 - 5 x + 6 = 0'
==> parse-latex() 
==> solve()

[3 2]

In [9]:
#%markdown
'x^2 - 6 x + 1 = 0'
==> parse-latex() 
==> solve()
==> to-latex()

$3+2\sqrt{2}, \: 3-2\sqrt{2}$

In [10]:
#% markdown
'x^2 - a x + 1 = 0'
==> parse-latex() 
==> solve(vars => 'a')
==> to-latex()

$x+\frac{1}{x}$

---

## Calculus

Derivate of expression over the variable $x$:

In [11]:
#%markdown
'D_x (x^2 + 1)'
==> parse-latex()
==> evaluate()
==> to-latex()

$2x$

Evaluate the undefined integral $\int x^2 dx$:

In [12]:
#%markdown
'\int x^2 dx'
==> parse-latex()
==> evaluate()
==> to-latex()

$\frac{x^3}{3}$

Evaluate the defined integral $\int_{0}^{1} x^2 dx$:

In [13]:
#%markdown
'\int_{0}^{1} x^2 dx'
==> parse-latex()
==> evaluate()
==> to-latex()

$\frac{1}{3}$

Evaluate the limit $\lim_{x \to 0} \frac{\sin(x)}{x}$:

In [42]:
'\lim_{x \to 0} \frac{\sin(x)}{x}'
==> parse-latex()
==> evaluate()
==> N()

1

---

## Function definition

See the original JavaScript examples [here](https://mathlive.io/compute-engine/guides/augmenting/).

In [14]:
evaluate(parse-latex('f(x) := 2x'));
evaluate(parse-latex('f(3)'));

6